# Spectral resolution

This offline tutorial compares the three Gaussian broadening laws. The narrow emission lines are analytic teaching data with negligible intrinsic width relative to the requested output widths.

In [ ]:
import astropy.units as u
import matplotlib.pyplot as plt
import numpy as np

from speclib import Spectrum

## A well-sampled input

The 0.05 Å sampling is much finer than every requested FWHM. `speclib` requires at least two input samples per target FWHM in every interval.

In [ ]:
wavelength = np.arange(4900.0, 5100.01, 0.05) * u.AA
values = np.zeros(wavelength.size)
for center in (4950.0, 5000.0, 5050.0):
    values += np.exp(-0.5 * ((wavelength.value - center) / 0.12) ** 2)
flux_unit = u.erg / (u.s * u.cm**2 * u.AA)
spectrum = Spectrum(spectral_axis=wavelength, flux=values * flux_unit)

## Constant wavelength FWHM

`set_spectral_resolution(3 * u.AA)` applies a 3 Å Gaussian FWHM everywhere. Its resolving power therefore increases with wavelength.

In [ ]:
constant_width = spectrum.set_spectral_resolution(3 * u.AA)

## Constant resolving power

At $R=1500$, the FWHM is $\lambda/R$: about 3.3 Å near 5000 Å. The convolution is evaluated in log wavelength using $\lambda F_\lambda$ so wavelength-integrated flux is the conserved measure away from finite boundaries.

In [ ]:
constant_r = spectrum.set_spectral_resolving_power(1500)

## Wavelength-dependent resolving power

The sampled curve must cover the entire spectrum. Resolving power is linearly interpolated between samples and should vary smoothly across a local resolution element.

In [ ]:
curve_wavelength = np.array([4900.0, 5000.0, 5100.0]) * u.AA
curve_r = np.array([1000.0, 1500.0, 2000.0])
variable_r = spectrum.set_variable_resolving_power(curve_wavelength, curve_r)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10, 3.2), sharey=True)
for ax, center in zip(axes, (4950.0, 5000.0, 5050.0)):
    region = np.abs(wavelength.value - center) < 8
    ax.plot(wavelength.value[region], spectrum.flux.value[region], color="0.7", label="input")
    ax.plot(wavelength.value[region], constant_width.flux.value[region], label=r"$\Delta\lambda=3$ A")
    ax.plot(wavelength.value[region], constant_r.flux.value[region], label=r"$R=1500$")
    ax.plot(wavelength.value[region], variable_r.flux.value[region], label=r"$R(\lambda)$")
    ax.set_xlabel("Wavelength [Angstrom]")
axes[0].set_ylabel("Flux density [arbitrary scale]")
axes[-1].legend(fontsize=8);

## Flux and sampling checks

All outputs retain the exact input axis. For these interior lines, the wavelength integral is conserved to numerical accuracy. Profiles at the finite boundaries have different guarantees: constant-width convolution extends endpoint values, whereas variable-width kernels are truncated and renormalized.

In [ ]:
for label, result in (("constant width", constant_width), ("constant R", constant_r), ("variable R", variable_r)):
    ratio = np.trapezoid(result.flux.value, wavelength.value) / np.trapezoid(spectrum.flux.value, wavelength.value)
    print(f"{label:14s}: same axis={np.array_equal(result.wavelength, spectrum.wavelength)}, integral ratio={ratio:.6f}")

Resolution methods do not deconvolve the input LSF and reject masks or uncertainties because no propagation rule is implemented. Resample separately if an extracted/detector wavelength grid is required.